In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

df = pd.read_csv("train.csv")
df.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [2]:
# melakukan prediksi sederhana
# kelompokkan terlebih dahulu

X = df.drop(columns=["PitNextLap"])
y = df.PitNextLap

X.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0


In [3]:
# pisahkan menggunakan split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

X_train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
215889,215889,D017,HARD,Pre-Season Testing,2023,0,28,2,10.0,12,92.878,-0.097,-20.820,0.538462,0.0
204847,204847,PEA,INTERMEDIATE,Canadian Grand Prix,2024,0,25,1,25.0,10,91.448,-28.820,-31.977,0.328947,3.0
118844,118844,D242,SOFT,Dutch Grand Prix,2022,0,6,1,7.0,2,77.664,-36.765,-45.569,0.083333,-2.0
258088,258088,D020,HARD,Las Vegas Grand Prix,2025,0,28,2,18.0,8,95.822,-1.800,-201.840,0.368421,4.0
90334,90334,HAD,HARD,Emilia Romagna Grand Prix,2024,0,38,2,21.0,7,82.534,-11.398,-136.539,0.493506,-3.0


In [4]:
# deteksi apakah ada kolom yang kosong pada train dan test
null_cols = [col for col in df.columns if df[col].isnull().any()]
print(null_cols)
# tidak terdapat column yang kosong
# bisa lewatkan untuk step 
print("tidak memiliki data yang kosong")

[]
tidak memiliki data yang kosong


In [5]:
# melakukan pembacaan kategori data
object_cols = X_train.select_dtypes(['object'])
# pisahkan antara high cardinality data dan low cardinality data
# low_cardinality_cols = [col for col in ]
print(object_cols.columns)
# mencari mana yang mempunyai low dan high cardinality
object_nunique = list(map(lambda col: X_train[col].nunique(), object_cols))
d = dict(zip(object_cols, object_nunique))

sorted(d.items(), key=lambda x:x[1])

Index(['Driver', 'Compound', 'Race'], dtype='str')


C:\Users\740195\AppData\Local\Temp\ipykernel_28388\166770103.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = X_train.select_dtypes(['object'])


[('Compound', 5), ('Race', 26), ('Driver', 875)]

In [6]:
# jadikan compound sebagai ordinal encoding
# mendefinisikan urutan encoder
compound_category = [['INTERMEDIATE', 'HARD', 'MEDIUM', 'SOFT', 'WET']]
my_ordinal_encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value', 
    unknown_value=-1,
    categories=compound_category
    )

X_train.head()
# melakukan drop terhadap data yang jelek
bad_label_cols = ['Driver']
good_label_cols_for_H = ['Compound']

label_X_train = X_train.drop(bad_label_cols, axis=1)
label_X_test = X_test.drop(bad_label_cols, axis=1)

print(display(label_X_train.head()))
# bagian driver sudah di drop

,id,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
215889,215889,HARD,Pre-Season Testing,2023,0,28,2,10.0,12,92.878,-0.097,-20.820,0.538462,0.0
204847,204847,INTERMEDIATE,Canadian Grand Prix,2024,0,25,1,25.0,10,91.448,-28.820,-31.977,0.328947,3.0
118844,118844,SOFT,Dutch Grand Prix,2022,0,6,1,7.0,2,77.664,-36.765,-45.569,0.083333,-2.0
258088,258088,HARD,Las Vegas Grand Prix,2025,0,28,2,18.0,8,95.822,-1.800,-201.840,0.368421,4.0
90334,90334,HARD,Emilia Romagna Grand Prix,2024,0,38,2,21.0,7,82.534,-11.398,-136.539,0.493506,-3.0


None


In [7]:
print(X_train['Compound'].unique())

<StringArray>
['HARD', 'INTERMEDIATE', 'SOFT', 'MEDIUM', 'WET']
Length: 5, dtype: str


In [8]:
# melakukan encoding ordinal pada good code
label_X_train[good_label_cols_for_H] = my_ordinal_encoder.fit_transform(X_train[good_label_cols_for_H])
label_X_test[good_label_cols_for_H] = my_ordinal_encoder.transform(X_test[good_label_cols_for_H])

print(display(label_X_train.head()))
# berhasil melakukan ordinal encoder yang sudah di siapkan, tinggal bagian race untuk di hot encoding

,id,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
215889,215889,1.0,Pre-Season Testing,2023,0,28,2,10.0,12,92.878,-0.097,-20.820,0.538462,0.0
204847,204847,0.0,Canadian Grand Prix,2024,0,25,1,25.0,10,91.448,-28.820,-31.977,0.328947,3.0
118844,118844,3.0,Dutch Grand Prix,2022,0,6,1,7.0,2,77.664,-36.765,-45.569,0.083333,-2.0
258088,258088,1.0,Las Vegas Grand Prix,2025,0,28,2,18.0,8,95.822,-1.800,-201.840,0.368421,4.0
90334,90334,1.0,Emilia Romagna Grand Prix,2024,0,38,2,21.0,7,82.534,-11.398,-136.539,0.493506,-3.0


None


In [9]:
# mulai bermain dengan data test
dft = pd.read_csv('test.csv')
display(dft.head())

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,439140,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
1,439141,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
2,439142,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
3,439143,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
4,439144,AND,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0


In [10]:
# mencoba menggunakan pipeline fungsinya
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
import warnings
warnings.filterwarnings("ignore")

dft = pd.read_csv("test.csv").drop(bad_label_cols, axis=1)

X_test_full = dft.copy()

# buat mesinnya
# ambil column yang di perlukan
all_numerical_cols = list(X_test_full.select_dtypes(["int64", "float64"]).columns)
print(all_numerical_cols)
all_object_col = list(X_test_full.select_dtypes(["object", "category"]).columns)
print(all_object_col)

# pisahkan beberapa
high_cardinal_cols = [cname for cname in all_object_col if X_test_full[cname].nunique() > 10]
low_cardinal_cols = [cname for cname in all_object_col if X_test_full[cname].nunique() < 10]

print(low_cardinal_cols)
print("ini adalah baris paling akhir")
# berhasil

['id', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']
['Compound', 'Race']
['Compound']
ini adalah baris paling akhir


In [12]:
# membuat mesin pipeline
from xgboost import XGBClassifier
from sklearn.preprocessing import OneHotEncoder
OH_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
low_cardinal_transformer = my_ordinal_encoder
high_cardinal_transformer = OH_encoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', all_numerical_cols),
        ('ord', low_cardinal_transformer, low_cardinal_cols),
        ('onehot', high_cardinal_transformer, high_cardinal_cols)
    ]
)

total_zero = (df['PitNextLap'] == 0).sum()
total_one = (df['PitNextLap'] == 1).sum()

imbalance_ratio = total_zero / total_one

my_model = XGBClassifier(
    n_estimators = 200,
    scale_pos_weight =imbalance_ratio,
    max_depth = 5,
    learning_rate = 0.05
)

# buat pipeline modelnya
my_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', my_model)
])

print("berhasil melakukan inisisasi pipeline")
print(my_pipeline)

berhasil melakukan inisisasi pipeline
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['id', 'Year', 'PitStop',
                                                   'LapNumber', 'Stint',
                                                   'TyreLife', 'Position',
                                                   'LapTime (s)',
                                                   'LapTime_Delta',
                                                   'Cumulative_Degradation',
                                                   'RaceProgress',
                                                   'Position_Change']),
                                                 ('ord',
                                                  OrdinalEncoder(categories=[['INTERMEDIATE',
                                                                              'HARD',
                                              

In [13]:
# melakukan prediksi

my_pipeline.fit(X, y)
final_test_prediction = my_pipeline.predict(X_test_full)

In [14]:
submission = pd.DataFrame({
    'id': dft['id'],
    'PitNextLap': final_test_prediction
})

# 4. Simpan ke dalam file CSV tanpa menyertakan indeks baris
submission.to_csv('submission.csv', index=False)

print("File 'submission.csv' berhasil dibuat!")

File 'submission.csv' berhasil dibuat!
